# Phase 9 — Evaluation

**Goal**: compare the rule-based selector vs. the Markov-aware selector across multiple metrics.

**Structure of this notebook**:

**Part 1 — Class-based Evaluation** uses only methods covered in PSAM 5020:
- Basic statistics (mean, std) — WK05, WK06
- Pearson correlation (`.corr()`) — WK05
- Recall / Precision — WK07

**Part 2 — Extended Evaluation** uses additional methods beyond the course:
- Wasserstein distance (`scipy.stats`) — distribution shape similarity
- Jensen-Shannon divergence (`scipy.spatial.distance`) — pattern distribution comparison
- Top-k local capture, size percentile — custom metrics
- Composite score — weighted combination

**Selectors compared**:
1. Current rule-based (`selected == 1`)
2. Markov-aware (`selected_markov == True`)

## 0. Setup

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance, pearsonr
from scipy.spatial.distance import jensenshannon

DATA_CLEAN_DIR = '../data_clean'
df = pd.read_csv(os.path.join(DATA_CLEAN_DIR, 'all_events_with_markov.csv'))
df = df.sort_values('t_rel').reset_index(drop=True)

df_all     = df.copy()
df_rule    = df[df['selected']==1].reset_index(drop=True)
df_markov  = df[df['selected_markov']].reset_index(drop=True)

print(f'All events    : {len(df_all):,}')
print(f'Rule selected : {len(df_rule):,}')
print(f'Markov sel.   : {len(df_markov):,}')

All events    : 4,312
Rule selected : 2,238
Markov sel.   : 2,430


## 9.1 Basic statistics of IEI
Compute mean and standard deviation of inter-event intervals for each selector. 
I will use the same `.mean()` and `.std()` methods as in WK05/06.

In [5]:
# Compute IEI for each selector
iei_all    = np.diff(df_all['t_rel'].values)
iei_rule   = np.diff(df_rule['t_rel'].values)
iei_markov = np.diff(df_markov['t_rel'].values)

# Make a DataFrame for tidy display (WK05 style)
stats = pd.DataFrame({
    'all':pd.Series(iei_all),
    'rule':pd.Series(iei_rule),
    'markov':pd.Series(iei_markov),
})

print('IEI statistics (in seconds):')
print(stats.describe().round(3))

IEI statistics (in seconds):
            all      rule    markov
count  4311.000  2237.000  2429.000
mean      3.705     7.134     6.575
std       7.164     9.037     9.072
min       0.000     0.062     0.000
25%       0.543     0.752     0.609
50%       0.623     3.262     2.464
75%       2.928     9.680     9.329
max      81.632    81.632    85.377


### Findings

## 9.2 Coefficient of variation
**CV = std / mean** is a normalized measure of spread. 
If CV is far from `all`, the rhythm has been over-regularized or made erratic.

In [6]:
def cv(data):
    if data.mean() == 0:
        return 0.0
    return data.std() / data.mean()

cv_all    = cv(iei_all)
cv_rule   = cv(iei_rule)
cv_markov = cv(iei_markov)

print(f'CV of IEI:')
print(f'  all     : {cv_all:.3f}  (this is the natural target)')
print(f'  rule    : {cv_rule:.3f}  (diff from all: {abs(cv_rule-cv_all):.3f})')
print(f'  markov  : {cv_markov:.3f}  (diff from all: {abs(cv_markov-cv_all):.3f})')

CV of IEI:
  all     : 1.933  (this is the natural target)
  rule    : 1.266  (diff from all: 0.667)
  markov  : 1.380  (diff from all: 0.554)


### Findings

## 9.3 Density time series correlation
Bin events into 5-second windows, count events per bin, then compute Pearson correlation 
with the unfiltered count series. 

If the selected stream rises and falls together with fermentation activity, correlation is high.

In [7]:
# Bin into 5-second windows
bin_sec = 5
edges = np.arange(0, df_all['t_rel'].max() + bin_sec, bin_sec)
h_all,    _ = np.histogram(df_all['t_rel'],    bins=edges)
h_rule,   _ = np.histogram(df_rule['t_rel'],   bins=edges)
h_markov, _ = np.histogram(df_markov['t_rel'], bins=edges)

# Build a DataFrame and use .corr() (WK05 style)
density_df = pd.DataFrame({
    'all':    h_all,
    'rule':   h_rule,
    'markov': h_markov,
})

print('Pearson correlation matrix (5s bins):')
print(density_df.corr().round(3))

Pearson correlation matrix (5s bins):
          all   rule  markov
all     1.000  0.803   0.894
rule    0.803  1.000   0.749
markov  0.894  0.749   1.000


In [8]:
# Try multiple bin sizes (1s, 5s, 30s) to see if correlation holds at different time scales
results = []
for B in [1, 5, 30]:
    edges = np.arange(0, df_all['t_rel'].max() + B, B)
    h_all_b,    _ = np.histogram(df_all['t_rel'],    bins=edges)
    h_rule_b,   _ = np.histogram(df_rule['t_rel'],   bins=edges)
    h_markov_b, _ = np.histogram(df_markov['t_rel'], bins=edges)
    
    df_b = pd.DataFrame({'all': h_all_b, 'rule': h_rule_b, 'markov': h_markov_b})
    corr_matrix = df_b.corr()
    
    results.append({
        'bin_sec': B,
        'rule_corr':   corr_matrix.loc['all', 'rule'],
        'markov_corr': corr_matrix.loc['all', 'markov'],
    })

print(pd.DataFrame(results).round(3))

   bin_sec  rule_corr  markov_corr
0        1      0.660        0.783
1        5      0.803        0.894
2       30      0.830        0.918


### Findings

## 9.4 Large bubble recall
**Recall** answers: of all the large bubbles in the data, what fraction did the selector capture?

I define "large bubbles" as those in the top 10% by area, then compute recall using 
the same `recall_score` style as WK07.


In [9]:
from sklearn.metrics import recall_score

# Define ground truth: 1 if the bubble is in the top 10% by area
for top_pct in [5, 10]:
    threshold = np.percentile(df_all['area'], 100 - top_pct)
    is_large = (df_all['area'] >= threshold).astype(int)
    rule_picked   = df_all['selected'].astype(int)
    markov_picked = df_all['selected_markov'].astype(int)
    
    # Recall = TP / (TP + FN) - of the truly large bubbles, how many did we catch?
    rule_recall   = recall_score(is_large, rule_picked   & is_large)
    markov_recall = recall_score(is_large, markov_picked & is_large)
    
    # Simpler version using set intersection
    large_ids = set(df_all[is_large == 1]['event_id'])
    rule_ids  = set(df_rule['event_id'])
    mark_ids  = set(df_markov['event_id'])
    rule_recall_simple   = len(large_ids & rule_ids) / len(large_ids) if large_ids else 0
    markov_recall_simple = len(large_ids & mark_ids) / len(large_ids) if large_ids else 0
    
    print(f'Top {top_pct}% bubbles by area (n={len(large_ids):,}):')
    print(f'  rule recall   : {rule_recall_simple:.3f}')
    print(f'  markov recall : {markov_recall_simple:.3f}')
    print()

Top 5% bubbles by area (n=223):
  rule recall   : 0.332
  markov recall : 0.596

Top 10% bubbles by area (n=451):
  rule recall   : 0.399
  markov recall : 0.605



### Findings

## Summary 

In [11]:
# Pull all Part 1 metrics into a summary table
part1 = pd.DataFrame({
    'metric': [
        'IEI mean (sec)',
        'IEI std (sec)',
        'CV (closer to all is better)',
        'Density corr B=5s',
        'Top 10% recall',
    ],
    'all':    [iei_all.mean(), iei_all.std(), cv_all, 1.0, 1.0],
    'rule':   [iei_rule.mean(),   iei_rule.std(),   cv_rule,
               density_df.corr().loc['all', 'rule'],
               len(set(df_rule['event_id']) &
                   set(df_all[df_all['area'] >=
                       np.percentile(df_all['area'], 90)]['event_id'])) /
               max(1, (df_all['area'] >=
                       np.percentile(df_all['area'], 90)).sum())],
    'markov': [iei_markov.mean(), iei_markov.std(), cv_markov,
               density_df.corr().loc['all', 'markov'],
               len(set(df_markov['event_id']) &
                   set(df_all[df_all['area'] >=
                       np.percentile(df_all['area'], 90)]['event_id'])) /
               max(1, (df_all['area'] >=
                       np.percentile(df_all['area'], 90)).sum())],
}).round(3)

print('Part 1 (class-based metrics) summary:')
print(part1.to_string(index=False))

Part 1 (class-based metrics) summary:
                      metric   all  rule  markov
              IEI mean (sec) 3.705 7.134   6.575
               IEI std (sec) 7.163 9.035   9.071
CV (closer to all is better) 1.933 1.266   1.380
           Density corr B=5s 1.000 0.803   0.894
              Top 10% recall 1.000 0.399   0.605


## 9.5 Audibilization